# Seizure beginning/end
optical vs LFP, TMEV or ChR2 stim (with window) including 1 min beforehand

In [ ]:
import sys
import os
import pandas as pd
notebook_dir = os.getcwd()
sys.path.insert(0, os.path.abspath(os.path.join(notebook_dir, '..')))  # Add the project root directory to the path
import custom_io as cio
import env_reader
import matplotlib.pyplot as plt
import h5py
import numpy as np
import scipy.io

In [ ]:
save_fig = False

In [ ]:
is_tmev = True # this changes automatically if ChR2 win stim is detected

In [ ]:
er = env_reader.read_env()
output_dir = er["OUTPUT_FOLDER"]

In [ ]:
def map_window_type(win_type: str):
    """
    Convert the window type to a standard naming convention

    Args:
        win_type (str): _description_

    Returns:
        _type_: _description_
    """
    if "ca1" in win_type.lower():
        return "CA1"
    if "cx" in win_type.lower() or "ctx" in win_type.lower() or "nc" in win_type.lower():
        return "CTX"

In [ ]:
fpath_assembled_traces = cio.open_file("Open assembled traces h5 file")
print(fpath_assembled_traces)

## TMEV

In [ ]:
dict_traces = dict()
with h5py.File(fpath_assembled_traces, "r") as f:
    for uuid in f.keys():
        # for win stim, only take sz + sd
        if "chr2" in f[uuid].attrs['exp_type'].lower():
            is_tmev = False
            if "sz" not in f[uuid].attrs['exp_type'].lower():
                continue
        has_lfp = f[uuid].attrs['has_lfp']
        n_recordings = len(f[uuid].attrs["recording_break_points"][()])
        if n_recordings > 1:  # each recording should have its own lfp
            has_lfp = has_lfp.all()
        segment_type_break_points_lfp = None if not has_lfp else f[uuid].attrs['segment_type_break_points_lfp'][()] 
        dict_traces[uuid] = {
            'mouse_id': f[uuid].attrs['mouse_id'],
            'window_type': map_window_type(f[uuid].attrs['window_type']),
            'has_lfp': has_lfp,
            'lfp_t': f[uuid]['lfp_t'][()],
            'lfp_y': f[uuid]['lfp_y'][()],
            'lfp_mov_y': f[uuid]['lfp_mov_y'][()],
            'lv_speed': f[uuid]['lv_speed'][()],
            'lv_t_s': f[uuid]['lv_t_s'][()],
            'mean_fluo': f[uuid]['mean_fluo'][()],
            'segment_type_break_points': f[uuid].attrs['segment_type_break_points'],
            'segment_type_break_points_lfp': segment_type_break_points_lfp,
        }
        if not is_tmev:
            dict_traces[uuid]["i_stim_begin_frame"] = f[uuid].attrs["i_stim_begin_frame"]
            dict_traces[uuid]["i_stim_end_frame"] = f[uuid].attrs["i_stim_end_frame"]
uuids_list = list(dict_traces.keys())

In [ ]:
def plot_fluo_lfp(uuid, t_before_sz_begin=60):
    has_lfp = dict_traces[uuid]['has_lfp']
    if not has_lfp:
        return
    i_begin_sz = dict_traces[uuid]['segment_type_break_points'][1]
    i_end_sz = dict_traces[uuid]['segment_type_break_points'][2]
    i_end_sz_lfp = dict_traces[uuid]['segment_type_break_points_lfp'][2]
    t_sz_begin = dict_traces[uuid]['lv_t_s'][i_begin_sz]
    # get 10 s before sz
    i_begin_plot =  dict_traces[uuid]['lv_t_s'].searchsorted(t_sz_begin - t_before_sz_begin)
    i_begin_plot_lfp = dict_traces[uuid]['lfp_t'].searchsorted(t_sz_begin - t_before_sz_begin)
    fluo_x = dict_traces[uuid]['lv_t_s'][i_begin_plot:i_end_sz]
    fluo_y = dict_traces[uuid]['mean_fluo'][i_begin_plot:i_end_sz]
    lfp_x = dict_traces[uuid]['lfp_t'][i_begin_plot_lfp:i_end_sz_lfp]
    lfp_y = dict_traces[uuid]['lfp_y'][i_begin_plot_lfp:i_end_sz_lfp]
    fig, axs = plt.subplots(2, 1, figsize=(15, 10), sharex=True)
    axs[0].plot(fluo_x, fluo_y, linewidth=0.5)
    axs[1].plot(lfp_x, lfp_y, linewidth=0.5, color="red")
    plt.suptitle(f"Mouse {dict_traces[uuid]['mouse_id']}, window type {dict_traces[uuid]['window_type']}")
    plt.show()

In [ ]:
dict_sz_begin_lfp_s_tmev = {
    "2251bba132cf45fa839d3214d1651392": 293.6,
    "7b9c17d8a1b0416daf65621680848b6a": 328.35,
    "9e75d7135137444492d104c461ddcaac": 328.25,
    "c7b29d28248e493eab02288b85e3adee": 328.25,
    "cd3c1e0e3c284a89891d2e4d9a7461f4": 315.3,
    "f5ccb81a34bb434482e2498bfdf88784": 325,
    }

dict_sz_begin_optical_s_tmev = {
    '2251bba132cf45fa839d3214d1651392': 328.3,
    '7b9c17d8a1b0416daf65621680848b6a': 328.24,
    '9e75d7135137444492d104c461ddcaac': 328.23,
    'c7b29d28248e493eab02288b85e3adee': 328.25,
    'cd3c1e0e3c284a89891d2e4d9a7461f4': 319.6,
    'f5ccb81a34bb434482e2498bfdf88784': 328,
}

dict_sz_end_optical_s_tmev = {
    "2251bba132cf45fa839d3214d1651392": 360.5,
    "7b9c17d8a1b0416daf65621680848b6a": 347.5,
    "9e75d7135137444492d104c461ddcaac": 342.7,
    "c7b29d28248e493eab02288b85e3adee": 347.2,
    "cd3c1e0e3c284a89891d2e4d9a7461f4": 390,
    "f5ccb81a34bb434482e2498bfdf88784": 367.5,
}

dict_sz_end_lfp_s_tmev = {
    "2251bba132cf45fa839d3214d1651392": 374,
    "7b9c17d8a1b0416daf65621680848b6a": 347.5,
    "9e75d7135137444492d104c461ddcaac": 343,
    "c7b29d28248e493eab02288b85e3adee": 347.5,
    "cd3c1e0e3c284a89891d2e4d9a7461f4": 396,
    "f5ccb81a34bb434482e2498bfdf88784": 367.7,
}


In [ ]:
dict_sz_begin_lfp_s = dict_sz_begin_lfp_s_tmev
dict_sz_begin_optical_s = dict_sz_begin_optical_s_tmev
dict_sz_end_optical_s = dict_sz_end_optical_s_tmev
dict_sz_end_lfp_s = dict_sz_end_lfp_s_tmev

In [ ]:
uuids_bad_lfp = []
for uuid in dict_traces.keys():
    if uuid not in dict_sz_begin_lfp_s.keys():
        uuids_bad_lfp.append(uuid)

n_bad_recordings_with_lfp = 0
for uuid in uuids_bad_lfp:
    if dict_traces[uuid]['has_lfp']:
        n_bad_recordings_with_lfp += 1

dict_sz_begin_optical_bad_lfp_s = {
    '30bcfb76a771468eab5c2a0bb71038d7': 328.17,
    '39f7ef9f661041428bdd57a5b15c7176': 328.2,
    '4dea78a01bf5408092f498032d67d84e': 328.27,
    '4e2310d2dde845b0908519b7196080e8': 328.21,
    '54c31c3151944cfd86043932d3a19b9a': 338.83,
    '58dbee01eacf4b7385e0192c812233da': 328.14,
    '5cfb012d47f14303a40680d2b333336a': 328.15,
    '5ecdd9dc8f13440f9becae3cde5ab574': 328.21,
    '74473c5d22e04525acf53f5a5cb799f4': 328.15,
    '7753b03a2a554cccaab42f1c0458d742': 339.66,
    'a39ed3a880c54f798eff250911f1c92f': 328.23,
    'aa66ae0470a14eb08e9bcadedc34ef64': 328.17,
    'd158cd12ad77489a827dab1173a933f9': 329.09,
    'f0442bebcd1a4291a8d0559eb47df08e': 361.26,
    'f481149fa8694621be6116cb84ae2d3c': 351.03
}

dict_sz_end_optical_bad_lfp_s = {
    '30bcfb76a771468eab5c2a0bb71038d7': 388.81,
    '39f7ef9f661041428bdd57a5b15c7176': 379.72,
    '4dea78a01bf5408092f498032d67d84e': 349,
    '4e2310d2dde845b0908519b7196080e8': 359.39,
    '54c31c3151944cfd86043932d3a19b9a': 363,
    '58dbee01eacf4b7385e0192c812233da': 376.97,
    '5cfb012d47f14303a40680d2b333336a': 391,
    '5ecdd9dc8f13440f9becae3cde5ab574': 364.77,
    '74473c5d22e04525acf53f5a5cb799f4': 392.67,
    '7753b03a2a554cccaab42f1c0458d742': 352,
    'a39ed3a880c54f798eff250911f1c92f': 364.46,
    'aa66ae0470a14eb08e9bcadedc34ef64': 348.92,
    'd158cd12ad77489a827dab1173a933f9': 369.21,
    'f0442bebcd1a4291a8d0559eb47df08e': 361.46,
    'f481149fa8694621be6116cb84ae2d3c': 359
}

In [ ]:
show_bad_lfp = False
if show_bad_lfp:
    n_recordings_bad_lfp = len(uuids_bad_lfp)
    fig, axs = plt.subplots(n_recordings_bad_lfp + n_bad_recordings_with_lfp, 1,figsize=(15, 10*n_recordings_bad_lfp))
    i_plot_bad_lfp = 0
    t_offset_bad_lfp = 5  # +- 5 s to plot
    for uuid in uuids_bad_lfp:
        t_sz_begin_optical = dict_sz_begin_optical_bad_lfp_s[uuid]
        t_sz_end_optical = dict_sz_end_optical_bad_lfp_s[uuid]
        t_begin_plot = t_sz_begin_optical - t_offset_bad_lfp
        t_end_plot = t_sz_end_optical + t_offset_bad_lfp
        has_lfp = dict_traces[uuid]['has_lfp']
        i_begin_plot = np.searchsorted(dict_traces[uuid]['lv_t_s'], t_begin_plot)
        i_end_plot = np.searchsorted(dict_traces[uuid]['lv_t_s'], t_end_plot)

        fluo_x = dict_traces[uuid]['lv_t_s'][i_begin_plot:i_end_plot]
        fluo_y = dict_traces[uuid]['mean_fluo'][i_begin_plot:i_end_plot]
        
        if has_lfp:
            i_begin_plot_lfp = np.searchsorted(dict_traces[uuid]['lfp_t'], t_begin_plot)
            i_end_plot_lfp = np.searchsorted(dict_traces[uuid]['lfp_t'], t_end_plot)
            lfp_x = dict_traces[uuid]['lfp_t'][i_begin_plot_lfp:i_end_plot_lfp]
            lfp_y = dict_traces[uuid]['lfp_y'][i_begin_plot_lfp:i_end_plot_lfp]
        axs[i_plot_bad_lfp].plot(fluo_x, fluo_y, linewidth=0.5, color="green")
        axs[i_plot_bad_lfp].set_title(f"{uuid}: {dict_traces[uuid]['mouse_id']}, {dict_traces[uuid]['window_type']}")
        axs[i_plot_bad_lfp].vlines([t_sz_begin_optical, t_sz_end_optical], ymin=np.min(fluo_y), ymax=np.max(fluo_y), color="green")  # optical seizure start, end
        axs[i_plot_bad_lfp].set_xlim([t_begin_plot, t_end_plot])
        i_plot_bad_lfp += 1
        if has_lfp:
            axs[i_plot_bad_lfp].plot(lfp_x, lfp_y, linewidth=0.5, color="blue")
            axs[i_plot_bad_lfp].set_xlim([t_begin_plot, t_end_plot])
            axs[i_plot_bad_lfp].vlines([t_sz_begin_optical, t_sz_end_optical], ymin=np.min(lfp_y), ymax=np.max(lfp_y), color="green")  # optical seizure start, end
            i_plot_bad_lfp += 1

    if save_fig:
        fpath_out = os.path.join(output_dir, "bad_lfp_tmev.pdf")
        plt.savefig(fpath_out, format="pdf", bbox_inches="tight")
        print(f"Saved to {fpath_out}")
    plt.show()

In [ ]:
n_seconds_from_sz = 141
n_seconds_before_sz = 60
fig, axs = plt.subplots(2*len(dict_sz_begin_lfp_s_tmev.keys()), 1, figsize=(18, 128))
ys_opto = []
xs_opto = []
ylims_min_opto = []
ylims_max_opto = []
ys_lfp = []
xs_lfp = []
uuids = []
mouse_ids = []
for i, uuid in enumerate(dict_sz_begin_lfp_s_tmev.keys()):
    t_lv = dict_traces[uuid]['lv_t_s'] - dict_sz_begin_lfp_s_tmev[uuid]
    t_lfp = dict_traces[uuid]['lfp_t'] - dict_sz_begin_lfp_s_tmev[uuid]
    idx_lv = t_lv >= -n_seconds_before_sz
    idx_lfp = t_lfp >= -n_seconds_before_sz
    # take 41 s (-1 to 40) of data
    t_lv = t_lv[idx_lv]
    t_lfp = t_lfp[idx_lfp]
    t_lv = t_lv[:(n_seconds_before_sz + n_seconds_from_sz)*15]
    t_lfp = t_lfp[:(n_seconds_before_sz + n_seconds_from_sz)*1000]
    mean_fluo = dict_traces[uuid]['mean_fluo'][idx_lv][:(n_seconds_before_sz + n_seconds_from_sz)*15]
    y_lfp = dict_traces[uuid]['lfp_y'][idx_lfp][:(n_seconds_before_sz + n_seconds_from_sz)*1000]
    # find max skipping stim range (which is much higher than Ca signal)
    y_max_opto = np.max(mean_fluo) + 5  # add some space to the top
    y_min_opto = np.min(mean_fluo)-0.5
    axs[2*i].plot(t_lv, mean_fluo, linewidth=0.5)
    axs[2*i].set_title(f"{uuid}: {dict_traces[uuid]['mouse_id']}, {dict_traces[uuid]['window_type']}")
    axs[2*i].set_xlim([-n_seconds_before_sz, n_seconds_from_sz-1])
    axs[2*i].set_ylim([y_min_opto, y_max_opto])
    axs[2*i+1].plot(t_lfp, y_lfp, linewidth=0.5, color="red")
    axs[2*i+1].set_xlim([-n_seconds_before_sz, n_seconds_from_sz-1])
    #axs[2*i+1].set_xticks(np.arange(-n_seconds_before_sz, n_seconds_from_sz, 1))
    #axs[2*i].set_xticks(np.arange(-n_seconds_before_sz, n_seconds_from_sz, 1))
    ys_opto.append(mean_fluo)
    xs_opto.append(t_lv)
    ys_lfp.append(y_lfp)
    xs_lfp.append(t_lfp)
    ylims_min_opto.append(y_min_opto)
    ylims_max_opto.append(y_max_opto)
    uuids.append(uuid)
    mouse_ids.append(dict_traces[uuid]['mouse_id'])
xs_opto = np.array(xs_opto)
ys_opto = np.array(ys_opto)
xs_lfp = np.array(xs_lfp)
ys_lfp = np.array(ys_lfp)
plt.savefig("D:\\tmev_sz_begin_end.pdf", format="pdf", bbox_inches="tight")
scipy.io.savemat("D:\\tmev_sz_begin_end.mat", {"ys_opto": ys_opto, "xs_opto": xs_opto, "ys_lfp": ys_lfp, "xs_lfp": xs_lfp, "ylims_min_opto": ylims_min_opto, "ylims_max_opto": ylims_max_opto, "uuids": uuids, "mouse_ids": mouse_ids})
plt.show()

## ChR2 win stim

In [ ]:
dict_traces = dict()
with h5py.File(fpath_assembled_traces, "r") as f:
    for uuid in f.keys():
        # for win stim, only take sz + sd
        if "chr2" in f[uuid].attrs['exp_type'].lower():
            is_tmev = False
            if "sz" not in f[uuid].attrs['exp_type'].lower():
                continue
        has_lfp = f[uuid].attrs['has_lfp']
        n_recordings = len(f[uuid].attrs["recording_break_points"][()])
        if n_recordings > 1:  # each recording should have its own lfp
            has_lfp = has_lfp.all()
        segment_type_break_points_lfp = None if not has_lfp else f[uuid].attrs['segment_type_break_points_lfp'][()] 
        dict_traces[uuid] = {
            'mouse_id': f[uuid].attrs['mouse_id'],
            'window_type': map_window_type(f[uuid].attrs['window_type']),
            'has_lfp': has_lfp,
            'lfp_t': f[uuid]['lfp_t'][()],
            'lfp_y': f[uuid]['lfp_y'][()],
            'lfp_mov_y': f[uuid]['lfp_mov_y'][()],
            'lv_speed': f[uuid]['lv_speed'][()],
            'lv_t_s': f[uuid]['lv_t_s'][()],
            'mean_fluo': f[uuid]['mean_fluo'][()],
            'segment_type_break_points': f[uuid].attrs['segment_type_break_points'],
            'segment_type_break_points_lfp': segment_type_break_points_lfp,
        }
        if not is_tmev:
            dict_traces[uuid]["i_stim_begin_frame"] = f[uuid].attrs["i_stim_begin_frame"]
            dict_traces[uuid]["i_stim_end_frame"] = f[uuid].attrs["i_stim_end_frame"]
uuids_list = list(dict_traces.keys())

In [ ]:
def plot_fluo_lfp(uuid, t_before_sz_begin=60):
    has_lfp = dict_traces[uuid]['has_lfp']
    if not has_lfp:
        return
    i_begin_sz = dict_traces[uuid]['segment_type_break_points'][1]
    i_end_sz = dict_traces[uuid]['segment_type_break_points'][2]
    i_end_sz_lfp = dict_traces[uuid]['segment_type_break_points_lfp'][2]
    t_sz_begin = dict_traces[uuid]['lv_t_s'][i_begin_sz]
    # get 10 s before sz
    i_begin_plot =  dict_traces[uuid]['lv_t_s'].searchsorted(t_sz_begin - t_before_sz_begin)
    i_begin_plot_lfp = dict_traces[uuid]['lfp_t'].searchsorted(t_sz_begin - t_before_sz_begin)
    fluo_x = dict_traces[uuid]['lv_t_s'][i_begin_plot:i_end_sz]
    fluo_y = dict_traces[uuid]['mean_fluo'][i_begin_plot:i_end_sz]
    lfp_x = dict_traces[uuid]['lfp_t'][i_begin_plot_lfp:i_end_sz_lfp]
    lfp_y = dict_traces[uuid]['lfp_y'][i_begin_plot_lfp:i_end_sz_lfp]
    fig, axs = plt.subplots(2, 1, figsize=(15, 10), sharex=True)
    axs[0].plot(fluo_x, fluo_y, linewidth=0.5)
    axs[1].plot(lfp_x, lfp_y, linewidth=0.5, color="red")
    plt.suptitle(f"Mouse {dict_traces[uuid]['mouse_id']}, window type {dict_traces[uuid]['window_type']}")
    plt.show()

In [ ]:

dict_sz_begin_lfp_s_chr2 = {
    "165df3ec480a4ef7adcc62735c850a1b": 300,
    "4ae789df9809469b8668ff01a8cc91ee": 300, 
    "77e5fc88100f4525bb827e1d0503460f": 300, 
    "8a6d1f27381b469c80e3cf72d4da9817": 300, 
    "8dec51d8e6944f97b07da4aa35c87e55": 300, 
    "904cc7c85915482c9fea5a43242fca5f": 300, 
    "9c9550fdbd15460b8aed0e87d8f6031e": 300, 
    "b9f18da25af3478caaccb17d87c0a4f4": 300, 
    "ea0966dfc987412c83b66c2535b9d622": 300, 
}

# the optical sz begin is already quantified. Optical sz end is not properly quantified, it was marking the appearance of SD waves (CA1) or a rough end (CTX)

dict_sz_begin_optical_s_chr2 = {
    "165df3ec480a4ef7adcc62735c850a1b": 310,
    "4ae789df9809469b8668ff01a8cc91ee": 310, 
    "77e5fc88100f4525bb827e1d0503460f": 310, 
    "8a6d1f27381b469c80e3cf72d4da9817": 310, 
    "8dec51d8e6944f97b07da4aa35c87e55": 310, 
    "904cc7c85915482c9fea5a43242fca5f": 310, 
    "9c9550fdbd15460b8aed0e87d8f6031e": 310, 
    "b9f18da25af3478caaccb17d87c0a4f4": 310, 
    "ea0966dfc987412c83b66c2535b9d622": 310,
}


dict_sz_end_optical_s_chr2 = {
    "165df3ec480a4ef7adcc62735c850a1b": 340,
    "4ae789df9809469b8668ff01a8cc91ee": 340, 
    "77e5fc88100f4525bb827e1d0503460f": 340, 
    "8a6d1f27381b469c80e3cf72d4da9817": 340, 
    "8dec51d8e6944f97b07da4aa35c87e55": 340, 
    "904cc7c85915482c9fea5a43242fca5f": 340, 
    "9c9550fdbd15460b8aed0e87d8f6031e": 340, 
    "b9f18da25af3478caaccb17d87c0a4f4": 340, 
    "ea0966dfc987412c83b66c2535b9d622": 340,
}



dict_sz_end_lfp_s_chr2 = {
    "165df3ec480a4ef7adcc62735c850a1b": 350,
    "4ae789df9809469b8668ff01a8cc91ee": 350, 
    "77e5fc88100f4525bb827e1d0503460f": 350, 
    "8a6d1f27381b469c80e3cf72d4da9817": 350, 
    "8dec51d8e6944f97b07da4aa35c87e55": 350, 
    "904cc7c85915482c9fea5a43242fca5f": 350, 
    "9c9550fdbd15460b8aed0e87d8f6031e": 350, 
    "b9f18da25af3478caaccb17d87c0a4f4": 350, 
    "ea0966dfc987412c83b66c2535b9d622": 350,
}

In [ ]:
dict_sz_begin_lfp_s = dict_sz_begin_lfp_s_chr2
dict_sz_begin_optical_s = dict_sz_begin_optical_s_chr2
dict_sz_end_optical_s = dict_sz_end_optical_s_chr2
dict_sz_end_lfp_s = dict_sz_end_lfp_s_chr2

In [ ]:
# overview plot
n_recordings = len(dict_sz_begin_lfp_s)
fig, axs = plt.subplots(2*n_recordings, 1,figsize=(15, 10*n_recordings))
i_plot = 0
t_offset = 10  # +- 5 s to plot
for uuid in dict_sz_begin_lfp_s.keys():
    i_begin_plot = dict_traces[uuid]['segment_type_break_points'][1]
    i_end_plot_lfp = dict_traces[uuid]['segment_type_break_points_lfp'][2]
    t_sz_end_optical = dict_sz_end_optical_s[uuid]
    t_sz_begin_optical = dict_sz_begin_optical_s[uuid]
    t_sz_begin_lfp = dict_sz_begin_lfp_s[uuid]
    t_sz_end_optical = dict_sz_end_optical_s[uuid]
    t_sz_end_lfp = dict_sz_end_lfp_s[uuid]
    i_end_plot = dict_traces[uuid]["lv_t_s"].searchsorted(t_sz_end_optical)

    t_begin_plot = min(t_sz_begin_optical, t_sz_begin_lfp) - t_offset
    t_end_plot = max(t_sz_end_optical, t_sz_end_lfp) + t_offset

    i_begin_plot = dict_traces[uuid]['lv_t_s'].searchsorted(t_begin_plot)
    i_end_plot = dict_traces[uuid]['lv_t_s'].searchsorted(t_end_plot)
    i_begin_plot_lfp = dict_traces[uuid]['lfp_t'].searchsorted(t_begin_plot)
    i_end_plot_lfp = dict_traces[uuid]['lfp_t'].searchsorted(t_end_plot)

    fluo_x = dict_traces[uuid]['lv_t_s'][i_begin_plot:i_end_plot]
    fluo_y = dict_traces[uuid]['mean_fluo'][i_begin_plot:i_end_plot]
    lfp_x = dict_traces[uuid]['lfp_t'][i_begin_plot_lfp:i_end_plot_lfp]
    lfp_y = dict_traces[uuid]['lfp_y'][i_begin_plot_lfp:i_end_plot_lfp]
    axs[i_plot].plot(fluo_x, fluo_y, linewidth=0.5, color="green")
    axs[i_plot].set_title(f"{uuid}: {dict_traces[uuid]['mouse_id']}, {dict_traces[uuid]['window_type']}")
    axs[i_plot].vlines([t_sz_begin_lfp, t_sz_end_lfp], ymin=np.min(fluo_y), ymax=np.max(fluo_y), color="blue")  # lfp seizure start, end
    axs[i_plot].vlines([t_sz_begin_optical, t_sz_end_optical], ymin=np.min(fluo_y), ymax=np.max(fluo_y), color="green")  # optical seizure start, end
    axs[i_plot].set_xlim([t_begin_plot, t_end_plot])
    i_plot += 1
    axs[i_plot].plot(lfp_x, lfp_y, linewidth=0.5, color="blue")
    axs[i_plot].vlines([t_sz_begin_lfp, t_sz_end_lfp], ymin=np.min(lfp_y), ymax=np.max(lfp_y), color="blue")  # lfp seizure start, end
    axs[i_plot].vlines([t_sz_begin_optical, t_sz_end_optical], ymin=np.min(lfp_y), ymax=np.max(lfp_y), color="green")  # optical seizure start, end
    axs[i_plot].set_xlim([t_begin_plot, t_end_plot])
    i_plot += 1
if save_fig:
    fpath_out = os.path.join(output_dir, "sz_begin_end_tmev.pdf")
    plt.savefig(fpath_out, format="pdf", bbox_inches="tight")
    print(f"Saved to {fpath_out}")
plt.show()

In [ ]:
# print the different times
for uuid in dict_sz_begin_lfp_s.keys():
    t_sz_begin_lfp = dict_sz_begin_lfp_s[uuid]
    t_sz_begin_optical = dict_sz_begin_optical_s[uuid]
    t_sz_end_lfp = dict_sz_end_lfp_s[uuid]
    t_sz_end_optical = dict_sz_end_optical_s[uuid]
    print(f"{uuid}:\noptical:\n\t{t_sz_begin_optical} - {t_sz_end_optical}\nlfp:\n\t{t_sz_begin_lfp} - {t_sz_end_lfp}\n")

In [ ]:
t_lfp_corrections_chr2 = {"165df3ec480a4ef7adcc62735c850a1b": 8.6,
"4ae789df9809469b8668ff01a8cc91ee": 0,
"77e5fc88100f4525bb827e1d0503460f": 0,
"8a6d1f27381b469c80e3cf72d4da9817": 8.5,
"8dec51d8e6944f97b07da4aa35c87e55": 8.5,
"904cc7c85915482c9fea5a43242fca5f": 0,
"9c9550fdbd15460b8aed0e87d8f6031e": 8.53,
"b9f18da25af3478caaccb17d87c0a4f4": 8.53,
"ea0966dfc987412c83b66c2535b9d622": 8.5}

In [ ]:
n_seconds_before_sz = 60
n_seconds_from_sz = 181
fig, axs = plt.subplots(2*len(dict_sz_begin_lfp_s_chr2.keys()), 1, figsize=(18, 128))
ys_opto = []
xs_opto = []
ylims_min_opto = []
ylims_max_opto = []
ys_lfp = []
xs_lfp = []
uuids = []
mouse_ids = []
for i, uuid in enumerate(dict_sz_begin_lfp_s_chr2.keys()):
    t_stim_begin = dict_traces[uuid]['lv_t_s'][dict_traces[uuid]["i_stim_begin_frame"]]
    t_lv = dict_traces[uuid]['lv_t_s'] - t_stim_begin
    t_lfp = dict_traces[uuid]['lfp_t'] + t_lfp_corrections_chr2[uuid] - t_stim_begin
    idx_lv = t_lv >= -n_seconds_before_sz
    idx_lfp = t_lfp >= -n_seconds_before_sz
    # take 41 s (-1 to 40) of data
    t_lv = t_lv[idx_lv]
    t_lfp = t_lfp[idx_lfp]
    t_lv = t_lv[:(n_seconds_before_sz+n_seconds_from_sz)*15]
    t_lfp = t_lfp[:(n_seconds_before_sz+n_seconds_from_sz)*1000]
    mean_fluo = dict_traces[uuid]['mean_fluo'][idx_lv][:(n_seconds_before_sz+n_seconds_from_sz)*15]
    y_lfp = dict_traces[uuid]['lfp_y'][idx_lfp][:(n_seconds_before_sz+n_seconds_from_sz)*1000]
    # find max skipping stim range (which is much higher than Ca signal)
    y_max_opto = np.max(dict_traces[uuid]['mean_fluo'][dict_traces[uuid]["i_stim_end_frame"]:]) + 5  # add some space to the top
    y_min_opto = np.min(mean_fluo)-0.5
    axs[2*i].plot(t_lv, mean_fluo, linewidth=0.5)
    axs[2*i].set_title(f"{uuid}: {dict_traces[uuid]['mouse_id']}, {dict_traces[uuid]['window_type']}")
    axs[2*i].set_xlim([-n_seconds_before_sz, n_seconds_from_sz-1])
    axs[2*i].set_ylim([y_min_opto, y_max_opto])
    axs[2*i+1].plot(t_lfp, y_lfp, linewidth=0.5, color="red")
    axs[2*i+1].set_xlim([-n_seconds_before_sz, n_seconds_from_sz-1])
    ys_opto.append(mean_fluo)
    xs_opto.append(t_lv)
    ys_lfp.append(y_lfp)
    xs_lfp.append(t_lfp)
    ylims_min_opto.append(y_min_opto)
    ylims_max_opto.append(y_max_opto)
    uuids.append(uuid)
    mouse_ids.append(dict_traces[uuid]['mouse_id'])
xs_opto = np.array(xs_opto)
ys_opto = np.array(ys_opto)
xs_lfp = np.array(xs_lfp)
ys_lfp = np.array(ys_lfp)
plt.savefig(f"{output_dir}\\opto_stim_sz_begin_end.pdf", format="pdf", bbox_inches="tight")
scipy.io.savemat(f"{output_dir}\\opto_stim_sz_begin_end.mat", {"ys_opto": ys_opto, "xs_opto": xs_opto, "ys_lfp": ys_lfp, "xs_lfp": xs_lfp, "ylims_min_opto": ylims_min_opto, "ylims_max_opto": ylims_max_opto, "uuids": uuids, "mouse_ids": mouse_ids})
plt.show()

# Cannula stim
plot all sz stims to include complete Sz

In [ ]:
dict_traces = dict()
with h5py.File(fpath_assembled_traces, "r") as f:
    for uuid in f.keys():
        # for cannula stim, only take sz + sd
        if "chr2" in f[uuid].attrs['exp_type'].lower():
            is_tmev = False
            if "sz" not in f[uuid].attrs['exp_type'].lower():
                continue
        i_stim_begin_frame = f[uuid].attrs['break_points_lfp'][()][1]
        t_lfp = f[uuid]['lfp_t'][()]
        y_lfp = f[uuid]['lfp_y'][()]
        #y_mov = f[uuid]['lfp_mov_y'][()]  # locomotion not needed now
        mouse_id = f[uuid].attrs['mouse_id']
        dict_traces[uuid] = {
            't_lfp': t_lfp,
            'y_lfp': y_lfp,
            #'y_mov': y_mov,
            'mouse_id': mouse_id,
            'i_stim_begin_frame': i_stim_begin_frame,
            'exp_type': f[uuid].attrs['exp_type'],
        }
        
        
        

In [ ]:
n_seconds_before_sz = 60
n_seconds_from_sz = 180
xs_lfp = []
ys_lfp = []
uuids = []
mouse_ids = []
exp_types = []
for uuid in dict_traces.keys():
    t_stim_begin = dict_traces[uuid]['t_lfp'][dict_traces[uuid]["i_stim_begin_frame"]]
    t_lfp = np.array(dict_traces[uuid]['t_lfp'] - t_stim_begin)
    y_lfp = np.array(dict_traces[uuid]['y_lfp'])
    # cut to specific size
    idx = t_lfp > -n_seconds_before_sz
    t_lfp = t_lfp[idx]
    t_lfp = t_lfp[:(n_seconds_before_sz + n_seconds_from_sz)*1000]
    y_lfp = y_lfp[idx]
    y_lfp = y_lfp[:(n_seconds_before_sz + n_seconds_from_sz)*1000]
    # add to lists
    xs_lfp.append(t_lfp)
    ys_lfp.append(y_lfp)
    uuids.append(uuid)
    mouse_ids.append(dict_traces[uuid]['mouse_id'])
    exp_types.append(dict_traces[uuid]['exp_type'])
xs_lfp = np.array(xs_lfp)
ys_lfp = np.array(ys_lfp)

In [ ]:
fig, axs  = plt.subplots(len(dict_traces.keys()), 1, figsize=(18, 128)) 
for i, uuid in enumerate(dict_traces.keys()):
    axs[i].plot(xs_lfp[i], ys_lfp[i], linewidth=0.5)
    axs[i].set_title(f"{uuid}: {dict_traces[uuid]['mouse_id']}, {dict_traces[uuid]['exp_type']}")
    #axs[i].set_xlim([-1, 60])
plt.savefig(f"{output_dir}\\cannula_stim_sz_begin_end.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
#save as Matlab workspace
scipy.io.savemat(f"{output_dir}\\cannula_stim_sz_begin_end.mat", {"ys_lfp": ys_lfp, "xs_lfp": xs_lfp, "uuids": uuids, "mouse_ids": mouse_ids, "exp_types": exp_types})